# Requirement 6 · Hebbian, modified Hebbian, competitive, error-correction learning and the Boltzmann machine

Each classic rule is applied to the project's own data and checked against a standard method:
eigenfaces (Hebb / Oja / Sanger vs SVD), grouping unknown faces (competitive learning vs k-means),
engagement from face pixels (perceptron and delta rule vs the MLP), and a Restricted Boltzmann Machine on faces.
See `docs/guide/06_classic_learning_rbm.md`. **CPU runtime is enough** (about 2 minutes).

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

In [ ]:
import os
if not os.path.exists('data/raw/drowsiness') or not os.path.exists('data/processed/lfw/emb_clean.npz'):
    !python -m smartattend stage data      # drowsiness frames + LFW
    !python -m smartattend stage face      # LFW face embeddings (about 20 minutes on CPU)

## 1 · Run everything

In [ ]:
!python -m smartattend classic

In [ ]:
from IPython.display import Image, display
for name in ['hebbian_eigenfaces', 'competitive_errorcorr_rbm', 'rbm_face_reconstructions']:
    display(Image(f'reports/figures/nnfund/{name}.png'))

## 2 · The rules in a few lines each

In [ ]:
import numpy as np
from smartattend.nnfund import classic_real as cr
X = np.load('data/processed/drowsiness_faces32.npz')['train_X'].reshape(-1, 1024)
Xc = X - X.mean(0); Xc /= np.sqrt(np.mean(np.sum(Xc**2, 1)))
w, norms = cr.oja_run(Xc, lr=0.01, epochs=5)
pc1 = np.linalg.svd(Xc, full_matrices=False)[2][0]
print('Oja |w| per epoch:', np.round(norms, 3), ' |cos| with SVD PC1:', round(abs(w @ pc1) / np.linalg.norm(w), 4))
_, hebb = cr.hebbian_run(Xc, lr=0.01, epochs=5)
print('Hebb |w| per epoch:', [f'{v:.1e}' for v in hebb])

## 3 · Tests

In [ ]:
!python -m pytest -q tests/test_classic_real.py tests/test_classic_and_rbm.py